# camel beauty assessor — frozen backbone sweep

every run here freezes part of the network instead of fine-tuning all of it. three freeze depths
across six datasets, plus one control that turns mosaic back on.

| axis | values |
|---|---|
| freeze depth | `11` backbone, `16` backbone and half the neck, `23` head only |
| augmentation | none, 90 degree turns, 45 degree turns |
| colour | rgb, grayscale |

mosaic is off, matching `fine-tuning_experiments.ipynb`, so these compare directly to `e0-nomosaic` and the rest.
`c-mosaic` is the single run that puts it back.

## 1. setup

### 1.1 imports

In [ ]:
import os
import csv
import time
import pathlib
import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import mlflow
from ultralytics import YOLO, settings as yolo_settings

print(f"torch {torch.__version__}   cuda {torch.cuda.is_available()}")

### 1.2 paths

In [ ]:
root = pathlib.Path.cwd()
VERSION = "v2"
detect = root / "dataset" / VERSION / "detect"

project = root / "runs" / VERSION
results = root / "results" / VERSION
weights_dir = root / "weights"

datasets = {
    "plain": detect / "plain",
    "plain-gray": detect / "gray",
    "rot90": detect / "rot90-rgb",
    "rot90-gray": detect / "rot90-gray",
    "rot45": detect / "rot45-rgb",
    "rot45-gray": detect / "rot45-gray",
}

### 1.3 display options

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 115)
pd.set_option("display.expand_frame_repr", False)

### 1.4 config

In [ ]:
seed = 42
splits = ["train", "val", "test"]
freezes = [11, 16, 23]

canon = ["Camel", "High_withers", "Large-head", "Large-lips", "Large-nose",
         "Large_hump", "Long-legs", "Long-neck", "Wide_body"]

epochs = 200
imgsz = 640
patience = 50
mosaic = 0.0

### 1.5 device

In [ ]:
device = 0 if torch.cuda.is_available() else "cpu"
batch = 16 if device != "cpu" else 8
workers = 8 if device != "cpu" else 2
amp = device != "cpu"

print(f"device {device}   batch {batch}   workers {workers}   amp {amp}")
print(f"cuda: {torch.cuda.get_device_name(0)}")

### 1.6 experiment tracking

In [ ]:
tracking_uri = "sqlite:///" + (root / "mlflow.db").resolve().as_posix()
os.environ["MLFLOW_TRACKING_URI"] = tracking_uri
os.environ["MLFLOW_EXPERIMENT_NAME"] = f"camel-beauty-cv-{VERSION}"
yolo_settings.update({"mlflow": True})

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment(f"camel-beauty-cv-{VERSION}")

print("tracking to", tracking_uri)

## 2. input check

the six datasets are built by `preprocessing.ipynb`, `baseline.ipynb` and `fine-tuning_experiments.ipynb`.
nothing is rebuilt here.

In [ ]:
missing = {k: v for k, v in datasets.items() if not (v / "train" / "images").is_dir()}
assert not missing, f"missing datasets {list(missing)}. run baseline.ipynb and experiments.ipynb first"

sizes = pd.DataFrame([{"dataset": k,
                       **{sp: len(os.listdir(v / sp / "images")) for sp in splits}}
                      for k, v in datasets.items()]).set_index("dataset")
sizes

## 3. what freezing costs

layers 0 to 10 are the backbone, 11 to 23 the head. a freeze value of `n` locks everything below
layer `n`.

In [ ]:
base = YOLO(str(weights_dir / "yolo11n.pt")).model
total = sum(p.numel() for p in base.parameters())

trainable = pd.DataFrame([
    {"freeze": fz,
     "frozen": (fr := sum(p.numel() for n, p in base.named_parameters()
                          if n.startswith("model.") and int(n.split(".")[1]) < fz)),
     "trainable": total - fr,
     "trainable_pct": round(100 * (total - fr) / total, 1)}
    for fz in [0] + freezes]).set_index("freeze")
trainable

## 4. the grid

each run writes its own folder, so a run whose weights already exist is skipped. the loop can be
stopped and restarted without losing what finished.

In [ ]:
def run_one(name, data, freeze, mos):
    ckpt = project / name / "weights" / "best.pt"
    started = time.time()

    if ckpt.exists():
        minutes = None
    else:
        YOLO(str(weights_dir / "yolo11n.pt")).train(
            data=str(data / "data.yaml"), project=str(project), name=name, exist_ok=True,
            epochs=epochs, imgsz=imgsz, batch=batch, device=device, workers=workers,
            seed=seed, deterministic=True, amp=amp, patience=patience, plots=True,
            mosaic=mos, freeze=freeze)
        minutes = round((time.time() - started) / 60, 1)

    return {"run": name, "freeze": freeze, "dataset": data.name, "mosaic": mos,
            "best": str(ckpt), "epochs": epochs, "imgsz": imgsz, "batch": batch,
            "device": str(device), "minutes": minutes}


plan = [(f"f{fz}-{tag}", datasets[tag], fz, mosaic) for fz in freezes for tag in datasets]
plan.append(("c-mosaic", datasets["plain"], freezes[0], 1.0))

print(f"{len(plan)} runs planned")
pd.DataFrame(plan, columns=["run", "dataset", "freeze", "mosaic"]).assign(
    dataset=lambda d: d.dataset.map(lambda p: p.name))

In [ ]:
records = []
for i, (name, data, fz, mos) in enumerate(plan, 1):
    print(f"[{i}/{len(plan)}] {name}")
    records.append(run_one(name, data, fz, mos))

done = pd.DataFrame(records).set_index("run")
done[["freeze", "dataset", "mosaic", "minutes"]]

## 5. learning curves

In [ ]:
def history(rec):
    path = pathlib.Path(rec["best"]).parents[1] / "results.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    return df


def loss_pair(df):
    train = df[[c for c in df.columns if c.startswith("train/") and c.endswith("_loss")]].sum(axis=1)
    val = df[[c for c in df.columns if c.startswith("val/") and c.endswith("_loss")]].sum(axis=1)
    return train, val


def map_col(df):
    return next(c for c in df.columns if "mAP50-95" in c)


trained = [r for r in records if history(r) is not None]
assert trained, "no results.csv found, run section 4 first"
print(f"{len(trained)} runs have curves")

In [ ]:
fig, axes = plt.subplots(1, len(freezes), figsize=(6 * len(freezes), 4.5), sharey=True)
for ax, fz in zip(np.atleast_1d(axes), freezes):
    for rec in trained:
        if rec["freeze"] != fz or rec["mosaic"]:
            continue
        df = history(rec)
        ax.plot(df.epoch, df[map_col(df)], label=rec["run"].split("-", 1)[1])
    ax.set_title(f"freeze = {fz}", fontsize=10)
    ax.set_xlabel("epoch")
    ax.legend(fontsize=8)
axes[0].set_ylabel("val mAP50-95")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for rec in trained:
    df = history(rec)
    train, val = loss_pair(df)
    ax.plot(df.epoch, val - train, label=rec["run"], lw=1)
ax.axhline(0, color="grey", lw=0.8)
ax.set_title("val loss minus train loss")
ax.set_xlabel("epoch")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for rec in trained:
    df = history(rec)
    train, val = loss_pair(df)
    pos = int(df[map_col(df)].values.argmax())
    rows.append({"run": rec["run"], "freeze": rec["freeze"], "epochs_run": int(df.epoch.max()),
                 "best_epoch": int(df.epoch.iloc[pos]),
                 "best_val_mAP50_95": round(float(df[map_col(df)].max()), 4),
                 "val_loss_best": round(float(val.min()), 3),
                 "val_loss_end": round(float(val.iloc[-1]), 3)})

curves = pd.DataFrame(rows).set_index("run")
curves["val_rebound"] = (curves.val_loss_end - curves.val_loss_best).round(3)
curves.sort_values("best_val_mAP50_95", ascending=False)

## 6. results

In [ ]:
def evaluate(rec, data, split="test"):
    m = YOLO(rec["best"]).val(data=str(data / "data.yaml"), split=split, imgsz=imgsz, batch=batch,
                              device=device, project=str(project), name=f"{rec['run']}-{split}",
                              exist_ok=True, plots=True)
    out = dict(rec)
    out.update({"split": split, "mAP50": round(float(m.box.map50), 4), "mAP50_95": round(float(m.box.map), 4),
                "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
                "val_dir": str(m.save_dir)})
    return out, m


tested, models = [], {}
for rec in records:
    data = next(v for v in datasets.values() if v.name == rec["dataset"])
    out, m = evaluate(rec, data)
    tested.append(out)
    models[rec["run"]] = m

compare = pd.DataFrame(tested).set_index("run")[
    ["freeze", "dataset", "mosaic", "minutes", "mAP50", "mAP50_95", "precision", "recall"]]
compare.sort_values("mAP50_95", ascending=False)

### 6.1 freeze depth against dataset

In [ ]:
grid = compare[compare.mosaic == 0].reset_index()
grid["aug"] = grid.run.str.split("-", n=1).str[1]
pivot = grid.pivot(index="aug", columns="freeze", values="mAP50_95")
pivot["best"] = pivot.max(axis=1)
pivot.sort_values("best", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
pivot.drop(columns="best").plot(kind="bar", ax=ax)
ax.set_ylabel("test mAP50-95")
ax.set_xlabel("")
ax.legend(title="freeze")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

### 6.2 against the fine-tuned runs

In [ ]:
prev = pd.read_csv(results / "experiments.csv").drop_duplicates("run", keep="last").set_index("run")
prev = prev[prev.imgsz == imgsz][["dataset", "mAP50", "mAP50_95", "precision", "recall"]]
prev["freeze"] = 0

both = pd.concat([prev, compare[["dataset", "mAP50", "mAP50_95", "precision", "recall", "freeze"]]])
both["vs e0"] = (both.mAP50_95 - prev.loc["e0-nomosaic", "mAP50_95"]).round(4)
both.sort_values("mAP50_95", ascending=False)

### 6.3 per class, best frozen run against the best fine-tuned one

In [ ]:
def per_class(m):
    rows = []
    for i, c in enumerate(list(m.box.ap_class_index)):
        p, r, ap50, ap = m.box.class_result(i)
        rows.append({"cls": canon[int(c)], "precision": round(float(p), 3), "recall": round(float(r), 3),
                     "mAP50": round(float(ap50), 3), "mAP50_95": round(float(ap), 3)})
    return pd.DataFrame(rows).set_index("cls").reindex(canon)


winner = compare.mAP50_95.idxmax()
pc = per_class(models[winner])
print(f"best frozen run: {winner}   mAP50-95 {compare.loc[winner, 'mAP50_95']}")
pc

## 7. mlops artifacts

In [ ]:
for rec in tested:
    with mlflow.start_run(run_name=rec["run"] + "-test"):
        mlflow.log_params({k: rec[k] for k in
                           ("freeze", "dataset", "mosaic", "epochs", "imgsz", "batch", "device", "split")})
        mlflow.log_metrics({k: rec[k] for k in ("mAP50", "mAP50_95", "precision", "recall")})
        for art in ("confusion_matrix_normalized.png", "PR_curve.png"):
            f = pathlib.Path(rec["val_dir"]) / art
            if f.exists():
                mlflow.log_artifact(str(f))

out = compare.reset_index()
out.insert(0, "logged_at", pd.Timestamp.now().isoformat(timespec="seconds"))
out.to_csv(results / "freezing.csv", index=False)
pc.to_csv(results / "per_class_frozen.csv")

print(f"{len(out)} rows written to {results / 'freezing.csv'}")
out